# Fake News Detection — QLoRA Fine-tuning (RoBERTa) + Multi-Agent Pipeline

Fine-tunes **`roberta-base`** for binary fake-news classification on the **LIAR** dataset using **QLoRA** (4-bit base + LoRA adapters), then builds a **LangGraph** multi-agent pipeline (RoBERTa classifier + FLAN-T5 explanation).

> **Note:** roberta-base is small (~125M params), so QLoRA gives little real memory benefit — plain LoRA would suffice. Used here as requested.
>
> **Fixes applied:** `device_map={"": 0}` (not `"auto"`) and `gradient_checkpointing=False` to avoid the bitsandbytes `AssertionError: module.weight.shape[1] == 1`.

In [ ]:
# Core libs
!pip install -q transformers datasets accelerate evaluate peft
# QLoRA needs bitsandbytes for 4-bit quantization
!pip install -q bitsandbytes
# Pin datasets (compat)
!pip install -q datasets==3.6.0
# Extras for the multi-agent pipeline (input parsing + search + graph)
!pip install -q scikit-learn scipy newspaper3k PyMuPDF duckduckgo-search langgraph lxml[html_clean]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 82.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.1 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 1. Load & inspect the LIAR data

In [ ]:
train_df = pd.read_csv("/content/train.tsv", sep='\t', header=None)
train_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


In [ ]:
train_df[1].unique()

array(['false', 'half-true', 'mostly-true', 'true', 'barely-true',
       'pants-fire'], dtype=object)

In [ ]:
print('train:', train_df.shape)

train: (10240, 14)


In [ ]:
test_df  = pd.read_csv("/content/test.tsv",  sep='\t', header=None)
valid_df = pd.read_csv("/content/valid.tsv", sep='\t', header=None)
print('valid:', valid_df.shape, '| test:', test_df.shape)

valid: (1284, 14) | test: (1267, 14)


## 2. Prepare dataset (binary mapping)

- **0 = real** → `mostly-true`, `true`
- **1 = fake** → `pants-fire`, `false`, `barely-true`, `half-true`

In [ ]:
import pandas as pd
from datasets import Dataset

def load_and_prepare_data(train_file, val_file, test_file):
    """Load LIAR TSVs and prepare binary classification dataset."""
    df_train = pd.read_csv(train_file, sep='\t', header=None)
    df_val   = pd.read_csv(val_file,   sep='\t', header=None)
    df_test  = pd.read_csv(test_file,  sep='\t', header=None)

    columns = [
        "id", "label_text", "statement", "subject", "speaker",
        "job_title", "state_info", "party_affiliation",
        "barely_true_count", "false_count", "half_true_count",
        "mostly_true_count", "pants_fire_count", "source"
    ]
    df_train.columns = columns
    df_val.columns   = columns
    df_test.columns  = columns

    def map_labels(label_text):
        fake_labels = ["pants-fire", "false", "barely-true", "half-true"]
        return 1 if label_text in fake_labels else 0

    for df in [df_train, df_val, df_test]:
        df['label'] = df['label_text'].apply(map_labels)
        df.dropna(subset=['label', 'statement'], inplace=True)

    return {
        'train':      Dataset.from_pandas(df_train[['statement', 'label']], preserve_index=False),
        'validation': Dataset.from_pandas(df_val[['statement', 'label']],   preserve_index=False),
        'test':       Dataset.from_pandas(df_test[['statement', 'label']],  preserve_index=False),
    }

In [ ]:
dataset = load_and_prepare_data(
    train_file="/content/train.tsv",
    val_file="/content/valid.tsv",
    test_file="/content/test.tsv"
)
print(dataset['train'][0])

import collections
print("train label counts:", collections.Counter(dataset['train']['label']))

{'statement': 'Says the Annies List political group supports third-trimester abortions on demand.', 'label': 1}
train label counts: Counter({1: 6602, 0: 3638})


## 3. Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["statement"], truncation=True,
                     padding="max_length", max_length=256)

tokenized_dataset = {
    split: dataset[split].map(tokenize_fn, batched=True)
    for split in ["train", "validation", "test"]
}

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

## 4. Load base model in 4-bit + apply QLoRA

**Fixed version:**
- `device_map={"": 0}` instead of `"auto"` (keeps 4-bit quant state intact on small models)
- `use_gradient_checkpointing=False` (gradient checkpointing is what triggered the bitsandbytes `AssertionError`)

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

num_labels = 2

# --- 4-bit quantization config (the "Q" in QLoRA) ---
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier"]   # ADD THIS too
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    quantization_config=bnb_config,
    device_map={"": 0}                     # FIX: pin to GPU 0 (not "auto")
)

# FIX: gradient checkpointing OFF -> avoids the 4-bit quant-state AssertionError
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=False
)

peft_config = LoraConfig(
    task_type="SEQ_CLS",
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "key", "value"]
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the che

trainable params: 1,476,866 || all params: 126,124,036 || trainable%: 1.1710


## 5. Metrics (classification-appropriate)

accuracy + macro-F1 + per-class (fake) precision/recall/F1 + ROC-AUC.
BLEU/ROUGE/METEOR/BERTScore are **not** used — those are for text generation, not classification.

In [ ]:
import evaluate
from scipy.special import softmax

accuracy  = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall    = evaluate.load("recall")
f1        = evaluate.load("f1")
roc_auc   = evaluate.load("roc_auc")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    preds = np.argmax(logits, axis=-1)
    pos_probs = probs[:, 1]  # probability of "fake"

    return {
        "accuracy":       accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro":       f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "precision_fake": precision.compute(predictions=preds, references=labels, average="binary", pos_label=1, zero_division=0)["precision"],
        "recall_fake":    recall.compute(predictions=preds, references=labels, average="binary", pos_label=1, zero_division=0)["recall"],
        "f1_fake":        f1.compute(predictions=preds, references=labels, average="binary", pos_label=1)["f1"],
        "roc_auc":        roc_auc.compute(prediction_scores=pos_probs, references=labels)["roc_auc"],
    }

## 6. Training

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results-liar",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    bf16=True,
    gradient_checkpointing=False,          # FIX: keep it off here too
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Fake,Recall Fake,F1 Fake,Roc Auc
1,0.639456,0.605165,0.680685,0.613298,0.737448,0.815972,0.774725,0.677719
2,0.612001,0.583460,0.690031,0.604905,0.729331,0.857639,0.788298,0.683722
3,0.590682,0.593724,0.665888,0.628568,0.762999,0.730324,0.746304,0.701491
4,0.573933,0.584714,0.687695,0.628373,0.748124,0.807870,0.776850,0.701667
5,0.555270,0.585672,0.687695,0.632658,0.752454,0.798611,0.774846,0.700324


TrainOutput(global_step=3200, training_loss=0.5942683219909668, metrics={'train_runtime': 2846.8174, 'train_samples_per_second': 17.985, 'train_steps_per_second': 1.124, 'total_flos': 6851788485427200.0, 'train_loss': 0.5942683219909668, 'epoch': 5.0})

In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])
print("\nTest set metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,Precision Fake,Recall Fake,F1 Fake,Roc Auc
0.555270,0.609009,5,0.660616,0.616203,0.720455,0.775061,0.746761,0.687480



Test set metrics:
  eval_loss: 0.6090
  eval_accuracy: 0.6606
  eval_f1_macro: 0.6162
  eval_precision_fake: 0.7205
  eval_recall_fake: 0.7751
  eval_f1_fake: 0.7468
  eval_roc_auc: 0.6875


## 7. Save the adapter

In [ ]:
save_path = "./fine_tuned_liar_detector"
model.save_pretrained(save_path)        # saves LoRA adapter (small)
tokenizer.save_pretrained(save_path)
print("Saved to", save_path)

Saved to ./fine_tuned_liar_detector


In [ ]:
from google.colab import files
import shutil
shutil.make_archive("fine_tuned_liar_detector", 'zip', "./fine_tuned_liar_detector")
files.download("fine_tuned_liar_detector.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Load models for inference

Load base in 4-bit again, then attach the saved LoRA adapter via `PeftModel`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16
# )

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier"]   # ADD THIS: head stays full-precision
)

roberta_tokenizer = AutoTokenizer.from_pretrained(save_path)

_base = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=2,
    quantization_config=bnb_config,
    device_map={"": 0}                      # FIX: not "auto"
)
roberta_model = PeftModel.from_pretrained(_base, save_path)
roberta_model.eval()
print("Classifier loaded (4-bit base + LoRA adapter).")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the che

Classifier loaded (4-bit base + LoRA adapter).


In [ ]:
# FLAN-T5 for explanations (used as-is, NOT fine-tuned)
from transformers import AutoModelForSeq2SeqLM

flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)
print("FLAN-T5 loaded.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded.


## 9. Input parser (Text | URL | PDF)

In [ ]:
import newspaper
import fitz  # PyMuPDF

def extract_text(input_type, value):
    try:
        if input_type == 'url':
            article = newspaper.Article(value)
            article.download()
            article.parse()
            return article.text
        elif input_type == 'pdf':
            with fitz.open(value) as doc:
                return "\n".join(page.get_text() for page in doc)
        elif input_type == 'text':
            return value
        else:
            return "Invalid input type"
    except Exception as e:
        return f"Error processing input: {str(e)}"

## 10. Multi-agent orchestration (LangGraph)

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import StateGraph, END
from duckduckgo_search import DDGS

In [ ]:
class AgentState(TypedDict):
    input_type: Literal["url", "pdf", "text"]
    value: str
    text: Optional[str]
    error: Optional[str]
    next: Optional[str]
    label: Optional[str]
    confidence: Optional[float]
    explanation: Optional[str]
    fallback_used: Optional[bool]

In [ ]:
def InputHandlerAgent(state: AgentState):
    text = extract_text(state["input_type"], state["value"])
    return {**state, "text": text}

def PlannerAgent(state: AgentState):
    if len(state.get("text", "").strip()) < 50:
        return {**state, "next": "FallbackSearchAgent"}
    return {**state, "next": "ToolRouterAgent"}

def FallbackSearchAgent(state: AgentState):
    query = state.get("value")
    try:
        result = next(DDGS().text(query))
        summary = result["body"]
    except Exception:
        summary = "No fallback results found."
    return {**state, "text": summary, "fallback_used": True}

def ToolRouterAgent(state: AgentState):
    return {**state, "next": "ExecutorAgent"}

In [ ]:
def ClassifierAgent(state: AgentState):
    inputs = roberta_tokenizer(
        state['text'], return_tensors="pt",
        truncation=True, padding='max_length', max_length=256
    ).to(roberta_model.device)
    with torch.no_grad():
        logits = roberta_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()
    label = "Real" if pred == 0 else "Fake"
    confidence = probs[0][pred].item()
    return {**state, "label": label, "confidence": confidence}

def ExplanationAgent(state: AgentState):
    prompt = f"Explain why this news might be {state['label'].lower()} in one sentence: {state['text']}"
    inputs = flan_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output_ids = flan_model.generate(inputs['input_ids'], max_new_tokens=100)
    explanation = flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return {**state, "explanation": explanation}

def ExecutorAgent(state: AgentState):
    state = ClassifierAgent(state)
    state = ExplanationAgent(state)
    return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("InputHandlerAgent", InputHandlerAgent)
graph.add_node("PlannerAgent", PlannerAgent)
graph.add_node("FallbackSearchAgent", FallbackSearchAgent)
graph.add_node("ToolRouterAgent", ToolRouterAgent)
graph.add_node("ExecutorAgent", ExecutorAgent)

graph.set_entry_point("InputHandlerAgent")
graph.add_edge("InputHandlerAgent", "PlannerAgent")
graph.add_conditional_edges("PlannerAgent", lambda s: s["next"], {
    "FallbackSearchAgent": "FallbackSearchAgent",
    "ToolRouterAgent": "ToolRouterAgent"
})
graph.add_edge("FallbackSearchAgent", "ToolRouterAgent")
graph.add_edge("ToolRouterAgent", "ExecutorAgent")
graph.add_edge("ExecutorAgent", END)

runnable = graph.compile()

## 11. Example usage

In [ ]:
result = runnable.invoke({
    "input_type": "text",
    "value": "The moon landing was staged in a Hollywood studio."
})

print("\nPrediction Results")
print(f"Label: {result['label']}")
print(f"Confidence: {result['confidence']:.2f}")
print(f"Explanation: {result['explanation']}")
if result.get("fallback_used"):
    print("Used fallback search")


Prediction Results
Label: Fake
Confidence: 0.93
Explanation: The moon landing was staged in a Hollywood studio.
